# Entrenamiento de los baselines en Colab

Se ejecuta de arriba abajo en una sesión limpia. Solo hay que editar la celda de configuración del
apartado 7, donde se elige qué modelo entrenar.

| # | Celda | Qué hace |
|---|---|---|
| 1 | Drive | Monta Drive, donde están el dataset y los checkpoints |
| 2 | Repositorio | Clona el repositorio |
| 3 | Dependencias | `pip install -r requirements.txt` y `pip install .` |
| 4 | Dataset | Descomprime la copia redimensionada desde Drive |
| 5 | Comprobación | Ejecuta `scripts/00_setup_check.py` |
| 6 | Sesión | Registra la GPU y los núcleos asignados |
| 7 | Entrenamiento | Lanza la campaña |

Cada celda comprueba su resultado y se detiene con un mensaje si algo falla.

## Preparación

1. El repositorio es público y se clona sin credenciales. Si se usa una copia privada, el token
   se guarda en los secretos de Colab como `GITHUB_TOKEN` (panel izquierdo, icono de la llave), nunca
   en una celda.

2. Dataset en Drive: sube `sh17_resized640.tar.gz` (unos 0,5 GB) a `MyDrive/tfm-edge/`. Se genera
   en local con:

   ```bash
   python scripts/02b_resize_dataset.py --target-size 640
   tar -C data -czf sh17_resized640.tar.gz resized_640
   ```

   Es la copia redimensionada, no el dataset original de 13 GB; entrenar sobre ella es unas 14 veces
   más rápido y no cambia lo que se mide. El archivo lleva su YAML y sus listas con rutas relativas.

## Reanudación

Un entrenamiento interrumpido no se puede reanudar. El entrenador conserva el modelo en memoria para
que la poda o los nodos de cuantización simulada no se pierdan, así que `setup_model` de Ultralytics
devuelve `None` y `resume_training` no hace nada; cargar `last.pt` reconstruiría el modelo desde el
YAML de arquitectura.

Los checkpoints por época quedan en Drive y se pueden usar como pesos, pero el entrenamiento no
continúa donde se quedó. Relanzar una ejecución interrumpida falla con el motivo; para rehacerla hay
que pasar `--no-resume`.

## 1. Montar Drive

Drive guarda el dataset y los checkpoints, que tienen que sobrevivir al cierre de la sesión. Una
campaña de varias horas puede superar la duración de la sesión, así que el entrenamiento escribe
directamente ahí.

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

DRIVE_BASE = Path("/content/drive/MyDrive/tfm-edge")
DRIVE_BASE.mkdir(parents=True, exist_ok=True)

DATASET_TAR = DRIVE_BASE / "sh17_resized640.tar.gz"
RUNS_DIR = DRIVE_BASE / "runs"
RUNS_DIR.mkdir(exist_ok=True)

assert DRIVE_BASE.is_dir(), f"No se ha montado Drive en {DRIVE_BASE}"
print(f"Drive montado. Base del proyecto: {DRIVE_BASE}")
print(f"  dataset esperado : {DATASET_TAR}  ({'presente' if DATASET_TAR.is_file() else 'FALTA'})")
print(f"  checkpoints en   : {RUNS_DIR}")

if not DATASET_TAR.is_file():
    disponibles = sorted(p.name for p in DRIVE_BASE.glob("*"))
    raise FileNotFoundError(
        f"Falta '{DATASET_TAR}'.\n"
        f"Lo que hay en {DRIVE_BASE}: {disponibles}\n"
        "Súbelo antes de continuar (ver la portada de este cuaderno)."
    )

Salida esperada: la ruta de Drive y el dataset como `presente`. Si aparece `FALTA`, la celda se
detiene y muestra el contenido del directorio.

## 2. Clonar el repositorio

El repositorio es público y se clona sin credenciales. Si existe el secreto `GITHUB_TOKEN`, se usa
en la URL de clonado y los mensajes de error de git se censuran antes de mostrarlos.

In [ ]:
import shutil
import subprocess

from google.colab import userdata

REPO_DIR = Path("/content/tfm-edge-object-detection")

# El repositorio es público, así que el secreto 'GITHUB_TOKEN' es opcional.
try:
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN") or ""
except Exception:
    GITHUB_TOKEN = ""


def _censurar(texto: str) -> str:
    """Quita el token de un texto antes de mostrarlo."""
    return texto.replace(GITHUB_TOKEN, "[TOKEN]") if GITHUB_TOKEN else texto


if REPO_DIR.is_dir():
    print(f"El repositorio ya está en {REPO_DIR}; se actualiza.")
    resultado = subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
        capture_output=True,
        text=True,
    )
else:
    resultado = subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            f"https://{GITHUB_TOKEN + '@' if GITHUB_TOKEN else ''}github.com/ajgomez23/tfm-compresion-yolov8.git",
            str(REPO_DIR),
        ],
        capture_output=True,
        text=True,
    )

if resultado.returncode != 0:
    raise RuntimeError(
        "Git ha fallado (mensaje censurado por si contenía el token):\n"
        + _censurar(resultado.stderr)[:800]
    )

# Se comprueba que el clon trae lo que hace falta, no solo que git dijera que si.
imprescindibles = [
    "pyproject.toml",
    "requirements.txt",
    "scripts/00_setup_check.py",
    "scripts/04_train_baselines.py",
    "src/tfm_edge",
]
faltan = [f for f in imprescindibles if not (REPO_DIR / f).exists()]
if faltan:
    raise RuntimeError(f"El clon está incompleto: faltan {faltan}")

revision = subprocess.run(
    ["git", "-C", str(REPO_DIR), "log", "-1", "--format=%h %s"],
    capture_output=True,
    text=True,
).stdout.strip()
print(f"Repositorio listo en {REPO_DIR}")
print(f"  revisión: {revision}")

Salida esperada: la revisión (hash y asunto del último commit), que identifica la versión del código
usada.

## 3. Instalar las dependencias

`requirements.txt` instala las versiones fijadas (Ultralytics incluido) y `pip install .` instala el
paquete `tfm_edge`. Se usa la instalación normal y no la editable porque en Colab `pip install -e .`
terminaba sin error pero `import tfm_edge` fallaba.

El `%cd` es necesario: `pip install .` instala el directorio actual.

In [ ]:
%cd /content/tfm-edge-object-detection
!pip install -q -r requirements.txt
!pip install -q .

In [ ]:
import importlib
import subprocess
import sys
import tempfile

# Se reinician los módulos ya cargados: pip puede haber sustituido versiones que
# esta sesión de Python tenía importadas de antes.
for modulo in [m for m in list(sys.modules) if m.startswith(("tfm_edge", "ultralytics"))]:
    del sys.modules[modulo]

fallos = []
for paquete, esperada in [("torch", None), ("ultralytics", "8.3.253")]:
    try:
        modulo = importlib.import_module(paquete)
        version = getattr(modulo, "__version__", "sin versión")
        marca = "OK" if not esperada or version == esperada else f"AVISO: se esperaba {esperada}"
        print(f"  {paquete:12s} {version:12s} {marca}")
    except Exception as error:
        fallos.append(f"{paquete}: {type(error).__name__}: {error}")

# 'import tfm_edge' se comprueba en un subproceso y desde un directorio neutro, porque
# el %cd ya ha añadido el repositorio a sys.path en este kernel.
with tempfile.TemporaryDirectory() as neutral:
    prueba = subprocess.run(
        [sys.executable, "-c", "import tfm_edge; print(tfm_edge.__file__)"],
        capture_output=True,
        text=True,
        cwd=neutral,
    )
if prueba.returncode == 0:
    print(f"  {'tfm_edge':12s} importable desde un intérprete limpio")
    print(f"               {prueba.stdout.strip()}")
else:
    fallos.append(
        "tfm_edge no es importable desde un intérprete limpio:\n      "
        + (prueba.stderr.strip().splitlines()[-1] if prueba.stderr else "sin detalle")
        + "\n      Reinstala con: !pip install --force-reinstall --no-deps ."
    )

import torch

print(f"  CUDA disponible: {torch.cuda.is_available()}")

if fallos:
    raise RuntimeError("Problemas de instalación:\n  - " + "\n  - ".join(fallos))
if not torch.cuda.is_available():
    raise RuntimeError(
        "No hay GPU en esta sesión. Cámbialo en Entorno de ejecución, Cambiar tipo de "
        "entorno de ejecución, GPU."
    )
print("\nDependencias correctas.")

Salida esperada: las versiones de torch y ultralytics, `tfm_edge` importable desde un intérprete
limpio y `CUDA disponible: True`.

In [ ]:
# requirements.txt fija numpy por debajo de 2.0 (lo exige el extra "export" de
# Ultralytics) y Colab trae la serie 2.x. pip baja la versión, pero los paquetes ya
# instalados siguen compilados contra la ABI anterior y pueden fallar después de
# importarse. Cada paquete se ejercita en un subproceso propio.
from tfm_edge.common.binary_compat import (
    check_numpy_binary_compat,
    format_binary_compat_report,
)

resultados = check_numpy_binary_compat()
print(format_binary_compat_report(resultados))

rotos = [r for r in resultados if r.installed and not r.healthy]
if rotos:
    raise RuntimeError(
        "Hay paquetes incompatibles con la NumPy instalada. Aplica el remedio de "
        "arriba y reinicia el entorno de ejecución antes de volver a ejecutar, porque "
        "el NumPy anterior sigue cargado en memoria."
    )

Salida esperada: las cinco líneas de la tabla con `[ OK ]`.

`[ ABI]` indica que el paquete está compilado contra otra versión de NumPy y la celda se detiene.
Hay que reiniciar el entorno de ejecución después de reinstalar. `[ -- ]` indica un paquete ausente,
que ya informa la celda anterior.

## 4. Descomprimir el dataset

El `.tar.gz` se descomprime en el disco local de la sesión. Leer miles de imágenes desde Drive haría
de la carga de datos el cuello de botella.

In [ ]:
import tarfile
import time

DATA_DIR = REPO_DIR / "data"
DATASET_DIR = DATA_DIR / "resized_640"
DATA_DIR.mkdir(exist_ok=True)

if DATASET_DIR.is_dir() and (DATASET_DIR / "sh17_ultralytics.yaml").is_file():
    print(f"El dataset ya está en {DATASET_DIR}; no se vuelve a descomprimir.")
else:
    inicio = time.perf_counter()
    print(f"Descomprimiendo {DATASET_TAR.name} ({DATASET_TAR.stat().st_size / 2**30:.2f} GiB)...")
    with tarfile.open(DATASET_TAR, "r:gz") as tar:
        tar.extractall(DATA_DIR)
    print(f"  hecho en {time.perf_counter() - inicio:.0f} s")

# Se comprueba el contenido, no solo que exista el directorio.
yaml_dataset = DATASET_DIR / "sh17_ultralytics.yaml"
num_imagenes = len(list((DATASET_DIR / "images").glob("*")))
num_etiquetas = len(list((DATASET_DIR / "labels").glob("*.txt")))

problemas = []
if not yaml_dataset.is_file():
    problemas.append(f"falta {yaml_dataset}")
if num_imagenes != 8099:
    problemas.append(f"hay {num_imagenes} imágenes y se esperaban 8.099")
if num_etiquetas != 8099:
    problemas.append(f"hay {num_etiquetas} etiquetas y se esperaban 8.099")
if problemas:
    raise RuntimeError("El dataset descomprimido no es correcto: " + "; ".join(problemas))

print(f"Dataset verificado: {num_imagenes:,} imágenes, {num_etiquetas:,} etiquetas")
print(f"  YAML: {yaml_dataset}")

In [ ]:
# Un tar.gz generado con una versión antigua puede llevar rutas absolutas.
from tfm_edge.data.prepare import assert_dataset_portable

try:
    assert_dataset_portable(DATASET_DIR)
    print("Portabilidad verificada: ninguna ruta absoluta en el YAML ni en las listas")
except Exception as error:
    raise RuntimeError(
        f"{error}\n\n"
        "El tar.gz de Drive se generó con rutas absolutas. "
        "Regenéralo en local y súbelo de nuevo:\n"
        "    python scripts/02b_resize_dataset.py --target-size 640 --force\n"
        "    tar -C data -czf sh17_resized640.tar.gz resized_640"
    ) from None

In [ ]:
# Ultralytics tiene que resolver las rutas contra esta ubicación.
from ultralytics.data.utils import check_det_dataset

datos = check_det_dataset(str(yaml_dataset), autodownload=False)
print(f"Ultralytics acepta el dataset: {datos['nc']} clases")
print(f"  train: {datos['train']}")
print(f"  val  : {datos['val']}")
assert datos["nc"] == 17, f"Se esperaban 17 clases y hay {datos['nc']}"
assert Path(datos["val"]).is_file(), "La lista de validación no se resuelve"
assert str(DATASET_DIR) in str(datos["val"]), (
    f"Las rutas apuntan fuera de esta sesión: {datos['val']}. " "El artefacto no es portable."
)
print("\nLas rutas se resuelven contra esta sesión, no contra la máquina de origen.")

Salida esperada: 8.099 imágenes, 17 clases y las rutas de `train` y `val` dentro de
`/content/.../data/resized_640/`. Si apuntan a otra máquina, hay que regenerar el `tar.gz`; la celda
anterior lo detecta e indica el comando.

## 5. Comprobación del entorno

`scripts/00_setup_check.py` comprueba versiones, dependencias, rutas y GPU. Su salida queda en el
cuaderno como registro del entorno de la sesión.

In [ ]:
!python scripts/00_setup_check.py

Salida esperada: el informe completo del entorno en el que se obtienen los resultados.

## 6. Máquina asignada

Colab asigna máquinas distintas en cada sesión y eso cambia los tiempos. La huella se guarda junto a
los checkpoints, en Drive.

In [ ]:
import json
import os
import subprocess

import torch


def _cores_fisicos() -> int:
    """Cuenta núcleos físicos leyendo /proc/cpuinfo."""
    try:
        texto = Path("/proc/cpuinfo").read_text()
        ids = {
            linea.split(":")[1].strip()
            for linea in texto.splitlines()
            if linea.startswith("core id")
        }
        return len(ids) or os.cpu_count() or 0
    except Exception:
        return os.cpu_count() or 0


gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
    capture_output=True,
    text=True,
).stdout.strip()

huella = {
    "gpu_nvidia_smi": gpu,
    "gpu_torch": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "",
    "gpu_memory_gib": (
        round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2)
        if torch.cuda.is_available()
        else ""
    ),
    "cpu_logicos": os.cpu_count(),
    "cpu_fisicos": _cores_fisicos(),
    "ram_gib": round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30, 1),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
}

for clave, valor in huella.items():
    print(f"  {clave:16s} {valor}")

destino_huella = RUNS_DIR / "session_environment.json"
historico = []
if destino_huella.is_file():
    historico = json.loads(destino_huella.read_text())
historico.append(huella)
destino_huella.write_text(json.dumps(historico, indent=2, ensure_ascii=False))
print(f"\nHuella añadida a {destino_huella} (sesión {len(historico)})")

Salida esperada: GPU, núcleos físicos y RAM. Con una T4 y 4 núcleos físicos, las campañas dieron
1,17 / 1,94 / 3,96 / 6,13 min por época de extremo a extremo para n / s / m / l. Con otra máquina
hay que volver a medir.

## 7. Entrenar

La salida va a Drive para que los checkpoints sobrevivan al cierre de la sesión. Escribir cada época
en Drive tiene un coste pequeño (el checkpoint ocupa entre 6 y 90 MB). Si la sesión se corta, el
entrenamiento se repite desde el principio, así que conviene revisar el presupuesto antes de lanzar.

Coste por época en Tesla T4, 640 px, lote 16, AMP y `workers=2`:

| Modelo | min/época (bucle) | min/época (extremo a extremo) | 100 épocas |
|---|---|---|---|
| yolov8n | sin medir | 1,17 | 1,95 h |
| yolov8s | 1,45 | 1,94 | 3,24 h |
| yolov8m | 2,94 | 3,96 | 6,60 h |
| yolov8l | 4,69 | 6,13 | 10,21 h |

Para presupuestar vale la tercera columna. La segunda sale de `measure_training_throughput.py` y solo
mide el bucle de entrenamiento; la tercera sale del `training_cost.json` de cada campaña e incluye la
validación, la escritura en Drive, el cierre del mosaico y la evaluación final. La diferencia es del
31 % al 35 %.

Las 10,21 h de `l` caben en el límite nominal de 12 h, pero no en las 10,2 h que quedan con el
margen de seguridad del 15 %, así que el planificador indica que no cabe y pide reanudación.

Se empieza por `s`, el baseline de las compresiones.

In [ ]:
# Configuración de la campaña.
TAMANOS = "s"  # "n", "s", "m" o "l". Varios: "n s m l"
EPOCAS = 100
LOTE = 16
REHACER = False  # True solo para rehacer una ejecución ya empezada


print(f"Salida en Drive: {RUNS_DIR}")
for size in TAMANOS.split():
    ultimo = RUNS_DIR / f"{size}_baseline_default" / "weights" / "last.pt"
    if not ultimo.is_file():
        estado = "empieza de cero"
    elif REHACER:
        estado = "se rehace desde cero y sustituye la ejecución anterior"
    else:
        estado = "hay una ejecución anterior: fallará (no se puede reanudar; pon REHACER=True)"
    print(f"  yolov8{size}: {estado}")

In [ ]:
# Se lanza con subprocess en lugar de '!' para no depender de cómo trata IPython
# las continuaciones de línea y para poder comprobar el código de salida.
import subprocess
import sys

comando = [
    sys.executable,
    "scripts/04_train_baselines.py",
    "--sizes",
    *TAMANOS.split(),
    "--epochs",
    str(EPOCAS),
    "--batch-size",
    str(LOTE),
    "--resolution",
    "640",
    "--workers",
    "2",
    "--output-dir",
    str(RUNS_DIR),
]
if REHACER:
    comando.append("--no-resume")
print(" ".join(comando), "\n")
# Sin capture_output: la salida va directa a la celda y se ve el avance en vivo.
codigo = subprocess.run(comando, cwd=REPO_DIR).returncode

if codigo == 0:
    print("\nEntrenamiento completado.")
elif codigo == 2:
    print(
        "\nAVISO: algún modelo ha fallado; los demás sí se entrenaron. "
        "Revisa el error más arriba."
    )
else:
    raise RuntimeError(
        f"El entrenamiento ha terminado con código {codigo}. Revisa el error "
        "más arriba. Si indica que no se puede reanudar, pon REHACER=True para "
        "rehacerla desde el principio o mueve el directorio para conservarla."
    )

Salida esperada: el plan de sesiones, el mAP final de cada modelo y las líneas
`Checkpoint verificado en FP32`, que confirman que los pesos no han pasado por media precisión.

El `training_cost.json` de la ejecución contiene el coste de entrenamiento (apartado 6.6) y tiene que
descargarse junto con `best.pt`.

## 8. Evaluar e inscribir en el registro

Al terminar el entrenamiento. Escribe en `results/registry/`, el registro de resultados del estudio.

In [ ]:
import subprocess
import sys

codigo = subprocess.run(
    [
        sys.executable,
        "scripts/05_evaluate_baselines.py",
        "--runs-dir",
        str(RUNS_DIR),
        "--resolution",
        "640",
        "--split",
        "val",
    ],
    cwd=REPO_DIR,
).returncode
if codigo not in (0, 2):
    raise RuntimeError(f"La evaluación ha terminado con código {codigo}.")

In [ ]:
# El registro vive en el disco de la sesión: hay que copiarlo a Drive antes de
# cerrarla, o se pierde.
import shutil

origen_registro = REPO_DIR / "results" / "registry"
destino_registro = DRIVE_BASE / "registry"
if origen_registro.is_dir():
    shutil.copytree(origen_registro, destino_registro, dirs_exist_ok=True)
    copiados = sorted(p.name for p in destino_registro.glob("*.csv"))
    print(f"Registro copiado a {destino_registro}: {copiados}")
else:
    print("Todavía no hay registro que copiar.")

Salida esperada: el resumen de validación del registro (filas por tabla y `Sin errores ni avisos`) y
los CSV copiados a Drive. Esos CSV se descargan para incorporarlos al registro local.